# TP Hadoop 1 à 4 — exécution dans Google Colab

Hadoop 3.5.0 en mode **pseudo-distribué** (1 NameNode, 1 DataNode, YARN), puis les TP 1 à 4 du dépôt.
Exécuter les cellules **dans l'ordre**. Après une déconnexion de Colab, tout reprendre depuis la cellule 0.

In [ ]:
# Cellule 0 - variables d'environnement (à rejouer après chaque redémarrage)
import os
HADOOP_VERSION = "3.5.0"
HADOOP_HOME = f"/content/hadoop-{HADOOP_VERSION}"
JAVA_HOME = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]
print("HADOOP_HOME =", HADOOP_HOME, "| JAVA_HOME =", JAVA_HOME)

In [ ]:
# Cellule 1 - Java 17, SSH, rsync, wget
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless openssh-server rsync wget > /dev/null
!java -version

In [ ]:
# Cellule 2 - Hadoop 3.5.0
!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz
!tar -xzf hadoop-3.5.0.tar.gz -C /content
!hadoop version | head -1

In [ ]:
%%bash
# Cellule 3 - configuration pseudo-distribuée (HDFS + YARN)
export HADOOP_HOME=/content/hadoop-3.5.0
sed -i 's|^# export JAVA_HOME=.*|export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64|' $HADOOP_HOME/etc/hadoop/hadoop-env.sh
echo 'export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64' >> $HADOOP_HOME/etc/hadoop/hadoop-env.sh
cat > $HADOOP_HOME/etc/hadoop/core-site.xml <<'EOF'
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>
EOF
cat > $HADOOP_HOME/etc/hadoop/hdfs-site.xml <<'EOF'
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.blocksize</name><value>134217728</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file:///content/hdfs/name</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file:///content/hdfs/data</value></property>
</configuration>
EOF
cat > $HADOOP_HOME/etc/hadoop/mapred-site.xml <<'EOF'
<configuration>
  <property><name>mapreduce.framework.name</name><value>yarn</value></property>
  <property><name>mapreduce.application.classpath</name><value>$HADOOP_HOME/share/hadoop/mapreduce/*:$HADOOP_HOME/share/hadoop/mapreduce/lib/*</value></property>
</configuration>
EOF
cat > $HADOOP_HOME/etc/hadoop/yarn-site.xml <<'EOF'
<configuration>
  <property><name>yarn.nodemanager.aux-services</name><value>mapreduce_shuffle</value></property>
  <property><name>yarn.nodemanager.env-whitelist</name><value>JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,CLASSPATH_PREPEND_DISTCACHE,HADOOP_YARN_HOME,HADOOP_HOME,PATH,LANG,TZ,HADOOP_MAPRED_HOME</value></property>
</configuration>
EOF
echo "configuration écrite"

In [ ]:
%%bash
# Cellule 4 - SSH, formatage (uniquement à l'installation) et démarrage
export HADOOP_HOME=/content/hadoop-3.5.0 JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export HDFS_NAMENODE_USER=root HDFS_DATANODE_USER=root HDFS_SECONDARYNAMENODE_USER=root YARN_RESOURCEMANAGER_USER=root YARN_NODEMANAGER_USER=root
service ssh start
mkdir -p ~/.ssh && (ssh-keygen -t rsa -P '' -f ~/.ssh/id_rsa >/dev/null 2>&1 || true)
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys && chmod 600 ~/.ssh/authorized_keys
ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null
rm -rf /content/hdfs
$HADOOP_HOME/bin/hdfs namenode -format -force -nonInteractive > /dev/null 2>&1
$HADOOP_HOME/sbin/start-dfs.sh
$HADOOP_HOME/sbin/start-yarn.sh
jps

In [ ]:
# Cellule 5 - vérifications (checklist)
!hdfs dfsadmin -report | grep -E "Live datanodes|Dead datanodes"
!hdfs dfsadmin -safemode get
!hdfs getconf -confKey dfs.blocksize
!hdfs getconf -confKey dfs.replication
!yarn node -list

In [ ]:
# Récupère les scripts du dépôt
!git clone -q https://github.com/TAGUE-GITH/bigdata-hadoop-spark.git /content/repo 2>/dev/null || git -C /content/repo pull -q
!mkdir -p /content/preuves_tp

## TP 1 — fichier de 300 Mo, réplication demandée 3 sur un seul DataNode

In [ ]:
!mkdir -p /content/tp1
!dd if=/dev/zero of=/content/tp1/logs_300Mo.bin bs=1M count=300 status=none
!ls -l /content/tp1/logs_300Mo.bin
!hdfs dfs -mkdir -p /tp1/input
!hdfs dfs -rm -f /tp1/input/logs_300Mo.bin
!hdfs dfs -Ddfs.blocksize=134217728 -Ddfs.replication=3 -put /content/tp1/logs_300Mo.bin /tp1/input/
!hdfs dfs -stat "Nom=%n Taille=%b Réplication=%r" /tp1/input/logs_300Mo.bin
!hdfs fsck /tp1/input/logs_300Mo.bin -files -blocks -locations | tee /content/preuves_tp/tp1_fsck.txt

In [ ]:
# Intégrité : les deux empreintes doivent être identiques
!md5sum /content/tp1/logs_300Mo.bin
!hdfs dfs -cat /tp1/input/logs_300Mo.bin | md5sum
# Calculs théoriques du fichier de 1 200 Mo
!python3 /content/repo/hadoop/tp1_architecte_hdfs/calculs_hdfs.py | head -20

In [ ]:
!hdfs dfs -rm -r -f /tp1
!rm -rf /content/tp1

## TP 2 — arborescence raw / curated / analytics / rejects

In [ ]:
!bash /content/repo/hadoop/tp2_admin_hdfs/tp2_hdfs.sh

## TP 3 — MapReduce Streaming avec compteurs de qualité

In [ ]:
!cd /content/repo/hadoop/tp3_mapreduce && bash test_local.sh
!cd /content/repo/hadoop/tp3_mapreduce && bash run_hadoop.sh

In [ ]:
# Q22 : provoquer FileAlreadyExistsException (sortie déjà présente), puis lire le message
!STREAMING_JAR=$(find $HADOOP_HOME/share/hadoop/tools/lib -name 'hadoop-streaming-*.jar' | head -1);  cd /content/repo/hadoop/tp3_mapreduce && hadoop jar $STREAMING_JAR -files mapper.py,reducer.py  -mapper "python3 mapper.py" -reducer "python3 reducer.py" -input /tp_avance/ca/input -output /tp_avance/ca/output 2>&1 | grep -m1 FileAlreadyExists

## TP 4 — petits fichiers et compaction

In [ ]:
!bash /content/repo/hadoop/tp4_petits_fichiers/tp4_smallfiles.sh